#Логит и пробит
###Как превратить «да или нет» в аккуратную вероятность


Модели логит и пробит оценивают вероятность да/нет

Они берут набор факторов (признаков)
и аккуратно переводят их в вероятность события:

купит / не купит

одобрит кредит / откажет

кликнет / проигнорирует

Проблема в том, что обычная линейная модель может предсказать что угодно —
−3, 2.7, 15…

Но вероятность не может быть меньше 0 или больше 1.

И вот здесь появляются логит и пробит —
они «сжимают» линейный прогноз в красивую S-образную кривую,
где результат всегда лежит между 0 и 1.
Логит и логистическая регрессия — это два понятия, которые неразрывно связаны.
Логит превращает вероятность в линейное число.
Сигмоида (логистическая функция) превращает линейное число в вероятность.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
from sklearn.linear_model import LinearRegression, LogisticRegression
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit, Probit
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import norm
from scipy.optimize import minimize

# Настройка отображения графиков
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
#ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЙ
print("="*80)
print("ЧАСТЬ 1: ВИЗУАЛИЗАЦИЯ РАСПРЕДЕЛЕНИЙ")
print("="*80)

# Создаем диапазон значений для z (линейного индекса)
z_range = np.linspace(-8, 8, 500)

# 1.1 Стандартное нормальное распределение (для пробита)
normal_pdf = stats.norm.pdf(z_range)      # плотность φ(z)
normal_cdf = stats.norm.cdf(z_range)      # функция распределения Φ(z)

# 1.2 Логистическое распределение (для логита)
logistic_cdf = expit(z_range)              # F(z) = 1/(1+e^(-z))
logistic_pdf = expit(z_range) * (1 - expit(z_range))  # производная f(z)

# Создаем фигуру с тремя подграфиками
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# График 1: Плотности распределений
axes[0].plot(z_range, normal_pdf, 'r-', linewidth=2.5, label='Нормальное (Пробит) φ(z)')
axes[0].plot(z_range, logistic_pdf, 'b--', linewidth=2.5, label='Логистическое (Логит) f(z)')
axes[0].fill_between(z_range, normal_pdf, alpha=0.2, color='red')
axes[0].fill_between(z_range, logistic_pdf, alpha=0.2, color='blue')
axes[0].set_xlabel('z (линейный индекс)')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Плотности распределений')
axes[0].legend()
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].plot(z_range, normal_cdf, 'r-', linewidth=2.5, label='Пробит: Φ(z)')
axes[1].plot(z_range, logistic_cdf, 'b--', linewidth=2.5, label='Логит: F(z) = 1/(1+e^(-z))')
axes[1].set_xlabel('z (линейный индекс)')
axes[1].set_ylabel('Вероятность P(Y=1) = F(z)')
axes[1].set_title('Функции распределения (CDF)')
axes[1].legend()
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[2].plot(z_range, normal_cdf, 'r-', linewidth=2, label='Пробит (нормальные хвосты)')
axes[2].plot(z_range, logistic_cdf, 'b--', linewidth=2, label='Логит (тяжелые хвосты)')
axes[2].set_xlabel('z (линейный индекс)')
axes[2].set_ylabel('Вероятность P(Y=1)')
axes[2].set_title('Сравнение хвостов распределений')
axes[2].set_xlim([2, 6])  # Увеличиваем правый хвост
axes[2].set_ylim([0.95, 1.0])
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Сравнение логистического и нормального распределений', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('01_distributions_comparison.png', dpi=150)
plt.show()

print("ВЫВОД 1:")
print("- Логистическое распределение имеет более тяжелые хвосты, чем нормальное")
print("- Это означает, что логит-модель придает больший вес экстремальным значениям")
print("- В центральной части (z от -2 до 2) распределения практически идентичны")
print("- Пробит использует нормальное распределение, логит - логистическое")

#Проблема: линейная модель и границы вероятности

В задачах бинарной классификации мы хотим предсказывать вероятность события:

𝑃(𝑦=1∣𝑥)

Если использовать обычную линейную регрессию, модель может выдавать любые числа — −2, 3.5, 10. Но вероятность по определению должна лежать в диапазоне от 0 до 1.

Кроме того, зависимость между признаками и вероятностью чаще всего нелинейная: влияние факторов усиливается или ослабевает по мере приближения к границе решения. Линейная модель этого не учитывает.

In [ ]:

print("\n" + "="*80)
print("ЧАСТЬ 2: ГЕНЕРАЦИЯ ДАННЫХ И ПРОБЛЕМЫ ЛИНЕЙНОЙ МОДЕЛИ")
print("="*80)

np.random.seed(42)
n_students = 1000

# Генерируем часы подготовки от 0 до 50
hours = np.random.uniform(0, 50, n_students)

# Истинная зависимость (как в лекции: β₁ = -9, β₂ = 0.5)
true_beta_0 = -9.0
true_beta_1 = 0.5
z_true = true_beta_0 + true_beta_1 * hours
p_true = expit(z_true)  # истинная вероятность

# Генерируем бинарный исход (сдал/не сдал)
passed = np.random.binomial(1, p_true)

# Создаем DataFrame
df = pd.DataFrame({
    'hours': hours,
    'passed': passed,
    'p_true': p_true
})

print(f"Сгенерировано {n_students} студентов")
print(f"Сдали зачет: {passed.sum()} ({passed.mean()*100:.1f}%)")
print(f"Среднее время подготовки: {hours.mean():.1f} часов")
print(f"Минимум часов: {hours.min():.1f}, Максимум: {hours.max():.1f}")

# Оцениваем линейную модель вероятности (ЛМВ)
lm = LinearRegression()
lm.fit(df[['hours']], df['passed'])
df['p_lm'] = lm.predict(df[['hours']])

# Визуализация проблем ЛМВ
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# График 1: Сравнение ЛМВ с истинной логистической кривой
hours_sorted = np.sort(df['hours'])
p_true_sorted = p_true[np.argsort(df['hours'])]
p_lm_sorted = lm.predict(hours_sorted.reshape(-1, 1))

axes[0].scatter(df['hours'], df['passed'], alpha=0.3, s=10, c='gray', label='Данные (0/1)')
axes[0].plot(hours_sorted, p_true_sorted, 'g-', linewidth=3, label='Истинная логистическая')
axes[0].plot(hours_sorted, p_lm_sorted, 'r--', linewidth=2, label='Линейная модель (ЛМВ)')
axes[0].axhline(y=0, color='k', linestyle=':', alpha=0.5)
axes[0].axhline(y=1, color='k', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Часы подготовки')
axes[0].set_ylabel('Вероятность сдачи / Факт сдачи')
axes[0].set_title('Сравнение ЛМВ с истинной моделью')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
# График 2: Демонстрация предсказаний вне [0,1]
# Создаем экстремальные значения для демонстрации
hours_extreme = np.array([-10, 0, 10, 20, 30, 40, 50, 60, 70])
p_lm_extreme = lm.predict(hours_extreme.reshape(-1, 1))

# Цвет: красный если вне [0,1], зеленый если внутри
colors = ['red' if p < 0 or p > 1 else 'green' for p in p_lm_extreme]

axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].axhline(y=1, color='k', linestyle='-', alpha=0.3)
axes[1].bar(range(len(hours_extreme)), p_lm_extreme, color=colors, alpha=0.7)
axes[1].set_xticks(range(len(hours_extreme)))
axes[1].set_xticklabels([f'{h:.0f}' for h in hours_extreme])
axes[1].set_xlabel('Часы подготовки')
axes[1].set_ylabel('Предсказанная вероятность')
axes[1].set_title('Проблема ЛМВ: предсказания вне [0,1]')
axes[1].fill_between([-1, 7], 0, 1, alpha=0.1, color='green', label='Допустимая область')
axes[1].legend()

plt.suptitle('Линейная модель вероятности (ЛМВ) и ее недостатки', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('02_lpm_problems.png', dpi=150)
plt.show()

print("\nВЫВОД 2:")
print("- ЛМВ может давать вероятности меньше 0 и больше 1 (некорректно)")
print("- При 60 часах подготовки ЛМВ предсказывает вероятность > 1")
print("- При 0 часах ЛМВ дает отрицательную вероятность")
print(
    "- Маржинальный эффект постоянный (β₂ = {:.3f}), что нереалистично. "
    "В реальности любой процесс имеет предел. "
    "Если студент готовился 0 часов, дополнительный час действительно может дать +2.7%? Возможно. "
    "Но если студент готовился 50 часов и уже имеет почти 100% вероятность сдачи, "
    "то дополнительный час уже не может дать +2.7% — это подняло бы вероятность выше 100%, "
    "что абсурдно.".format(lm.coef_[0])
)

#Оценка логит-модели

Логит-модель оценивается методом максимального правдоподобия (Maximum Likelihood Estimation, MLE). Поскольку зависимая переменная принимает значения 0 или 1, мы моделируем не сами значения, а вероятность наступления события. Для каждого наблюдения задаётся вероятность

𝑝𝑖=1/1+𝑒**−𝑥𝑖𝑇𝛽
	​

, и далее строится функция правдоподобия — вероятность получить именно те данные, которые мы наблюдаем, при заданных параметрах модели.
#Логит-модель: пример
pi = P(Yi = 1) = F(β1 + β2xi)

Влияние времени, затраченного на подготовку,
на вероятность сдачи зачёта
Yi - переменная, которая равна единице,
если i-й студент сдал зачёт, и нулю в противном случае
xi - время, затраченное i-м студентом на подготовку
pi = P(Yi = 1) - вероятность того, что зачёт будет успешно сдан


In [ ]:
print("\n" + "="*80)
print("ЧАСТЬ 3: ОЦЕНКА ЛОГИТ МОДЕЛИ")
print("="*80)

# Подготовка данных
X = sm.add_constant(df['hours'])
y = df['passed']

# Оценка логит модели
logit_model = Logit(y, X)
logit_result = logit_model.fit(disp=0)

print("\nРЕЗУЛЬТАТЫ ЛОГИТ МОДЕЛИ:")
print("-"*40)
print(f"Константа (β₁): {logit_result.params.iloc[0]:.3f}")
print(f"Коэффициент при часах (β₂): {logit_result.params.iloc[1]:.3f}")
print(f"Стандартная ошибка β₂: {logit_result.bse.iloc[1]:.4f}")
print(f"z-статистика: {logit_result.tvalues.iloc[1]:.3f}")
print(f"p-значение: {logit_result.pvalues.iloc[1]:.6f}")

# Предсказанные вероятности
df['p_logit'] = logit_result.predict()

# Расчет для конкретных студентов (как на слайде 15)
test_students = [5, 15, 25, 40]
print("\nВЕРОЯТНОСТИ ДЛЯ КОНКРЕТНЫХ СТУДЕНТОВ:")
print("-"*50)
print(f"{'Часы':^10} | {'z = β₁+β₂x':^15} | {'P = 1/(1+e^(-z))':^20} | {'Вероятность':^12}")
print("-"*65)

for h in test_students:
    z = logit_result.params.iloc[0] + logit_result.params.iloc[1] * h
    p = expit(z)
    print(f"{h:^10} | {z:^15.3f} | {p:^20.3f} | {p:^12.1%}")

# Студент из примера в лекции (15 часов)
h_lecture = 15
z_lecture = logit_result.params.iloc[0] + logit_result.params.iloc[1] * h_lecture
p_lecture = expit(z_lecture)
print(f"\nСтудент с {h_lecture} часами (как в лекции): P = {p_lecture:.1%}")

# Визуализация логит модели
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# График 1: Логит кривая с данными
hours_range = np.linspace(0, 50, 200)
X_range = sm.add_constant(hours_range)
p_range = logit_result.predict(X_range)

axes[0].scatter(df['hours'], df['passed'], alpha=0.2, s=5, c='gray', label='Данные')
axes[0].plot(hours_range, p_range, 'b-', linewidth=3, label='Логит модель')
axes[0].set_xlabel('Часы подготовки')
axes[0].set_ylabel('Вероятность сдачи P(Y=1)')
axes[0].set_title('Логит модель: P = 1/(1+e^(-({:.2f}{:+.2f}x)))'.format(
    logit_result.params.iloc[0], logit_result.params.iloc[1]))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График 2: Связь между z и вероятностью
z_values = logit_result.params.iloc[0] + logit_result.params.iloc[1] * hours_range

axes[1].plot(z_values, p_range, 'b-', linewidth=3)
axes[1].set_xlabel('z = β₁ + β₂x')
axes[1].set_ylabel('P(Y=1) = 1/(1+e^(-z))')
axes[1].set_title('Логистическая функция: трансформация z в вероятность')
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_logit_model.png', dpi=150)
plt.show()

print("\nВЫВОД 3:")
print("- Логит модель гарантирует вероятности в интервале (0, 1)")
print("- Форма кривой S-образная (сигмоида)")
print("- При z=0 вероятность = 0.5")
print("- При увеличении z вероятность асимптотически стремится к 1")

#ПРЕДЕЛЬНЫЕ ЭФФЕКТЫ В ЛОГИТ-МОДЕЛИ
В логит-модели коэффициенты интерпретируются как изменение логарифма шансов (log-odds), но на практике нас чаще интересует влияние признака на саму вероятность события. Предельный эффект показывает, как изменяется вероятность при увеличении признака на единицу.

В отличие от линейной регрессии, предельный эффект в логит-модели не является постоянным: он зависит от значения признаков.

Формально он равен
𝛽𝑗⋅𝑝(𝑥)(1−𝑝(𝑥))

Это означает, что влияние переменной максимально в области средних вероятностей (около 0.5) и уменьшается при приближении вероятности к 0 или 1. Поэтому часто рассчитывают средние предельные эффекты (Average Marginal Effects, AME).

In [ ]:
print("\n" + "="*80)
print("ЧАСТЬ 4: ПРЕДЕЛЬНЫЕ ЭФФЕКТЫ В ЛОГИТ МОДЕЛИ")
print("="*80)

def marginal_effect_logit(hours, coef):
    """Предельный эффект для логит модели: dp/dx = f(z) * β₂"""
    coef = np.asarray(coef)
    z = coef[0] + coef[1] * hours
    # f(z) - производная логистической функции = p*(1-p)
    p = expit(z)
    f_z = p * (1 - p)
    me = f_z * coef[1]
    return me, p, f_z

print("ПРЕДЕЛЬНЫЕ ЭФФЕКТЫ ДЛЯ РАЗНЫХ СТУДЕНТОВ:")
print("-"*70)
print(f"{'Часы':^8} | {'P(Y=1)':^10} | {'f(z)=p(1-p)':^12} | {'β₂':^6} | {'ME = f(z)*β₂':^15} | {'Интерпретация':^25}")
print("-"*70)

for h in test_students:
    me, p, f_z = marginal_effect_logit(h, logit_result.params)
    effect_pct = me * 100
    print(f"{h:^8} | {p:^10.3f} | {f_z:^12.4f} | {logit_result.params.iloc[1]:^6.2f} | {me:^15.4f} | +1 час = +{effect_pct:.2f}%")

# Демонстрация различия эффектов (как на слайде 18)
h1, h2 = 15, 40
me1, p1, _ = marginal_effect_logit(h1, logit_result.params)
me2, p2, _ = marginal_effect_logit(h2, logit_result.params)

print(f"\nСРАВНЕНИЕ ЭФФЕКТОВ (как в лекции):")
print(f"Студент с {h1} часами (P={p1:.1%}): ME = {me1:.4f} (+{me1*100:.2f}%)")
print(f"Студент с {h2} часами (P={p2:.1%}): ME = {me2:.8f} (эффект почти 0)")

# Визуализация предельных эффектов
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Рассчитываем ME для всего диапазона
me_range = [marginal_effect_logit(h, logit_result.params)[0] for h in hours_range]
f_z_range = [marginal_effect_logit(h, logit_result.params)[2] for h in hours_range]

# График 1: Предельный эффект
axes[0].plot(hours_range, me_range, 'b-', linewidth=3)
axes[0].set_xlabel('Часы подготовки')
axes[0].set_ylabel('Предельный эффект (dp/dx)')
axes[0].set_title('Предельный эффект дополнительного часа подготовки')
axes[0].axvline(x=h1, color='red', linestyle='--', alpha=0.7, label=f'{h1} ч')
axes[0].axvline(x=h2, color='orange', linestyle='--', alpha=0.7, label=f'{h2} ч')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График 2: Компоненты предельного эффекта
axes[1].plot(hours_range, f_z_range, 'g-', linewidth=2, label='f(z) = p(1-p)')
axes[1].axhline(y=logit_result.params.iloc[1], color='purple', linestyle='-',
                linewidth=2, label=f'β₂ = {logit_result.params.iloc[1]:.2f}')
axes[1].fill_between(hours_range, 0, np.array(me_range), alpha=0.3, color='blue',
                      label='ME = f(z)*β₂')
axes[1].set_xlabel('Часы подготовки')
axes[1].set_ylabel('Значение')
axes[1].set_title('Компоненты предельного эффекта')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Предельные эффекты в логит модели', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('04_marginal_effects.png', dpi=150)
plt.show()

print("\nВЫВОД 4:")
print("- Предельные эффекты не постоянны, а зависят от текущего значения x")
print("- Максимальный эффект достигается при P≈0.5 (в точке перегиба)")
print("- На краях распределения (P близко к 0 или 1) эффект минимален")
print("- Это отражает реальность: первый час подготовки важен, сотый - нет")


#ОЦЕНКА ПРОБИТ-МОДЕЛИ
Пробит-модель, как и логит, оценивается методом максимального правдоподобия. Вероятность события задаётся через кумулятивную функцию стандартного нормального распределения:

𝑃(𝑦=1∣𝑥)=Φ(𝑥𝑇𝛽)

Функция правдоподобия строится аналогично логит-модели, но вместо логистической функции используется нормальная CDF. Закрытого решения также не существует, поэтому параметры оцениваются численно. Отрицательное лог-правдоподобие служит функцией потерь, аналогично логистической регрессии.

In [ ]:
print("\n" + "="*80)
print("ЧАСТЬ 5: ОЦЕНКА ПРОБИТ МОДЕЛИ")
print("="*80)

# Оценка пробит модели
probit_model = Probit(y, X)
probit_result = probit_model.fit(disp=0)

print("\nРЕЗУЛЬТАТЫ ПРОБИТ МОДЕЛИ:")
print("-"*40)
print(f"Константа (β₁): {probit_result.params.iloc[0]:.3f}")
print(f"Коэффициент при часах (β₂): {probit_result.params.iloc[1]:.3f}")
print(f"Стандартная ошибка β₂: {probit_result.bse.iloc[1]:.4f}")
print(f"z-статистика: {probit_result.tvalues.iloc[1]:.3f}")
print(f"p-значение: {probit_result.pvalues.iloc[1]:.6f}")

# Предсказанные вероятности
df['p_probit'] = probit_result.predict()

def marginal_effect_probit(hours, coef):
    """Предельный эффект для пробит модели: dp/dx = φ(z) * β₂"""
    coef = np.asarray(coef)
    z = coef[0] + coef[1] * hours
    phi_z = stats.norm.pdf(z)  # плотность нормального распределения
    me = phi_z * coef[1]
    return me, stats.norm.cdf(z), phi_z

print("\nСРАВНЕНИЕ ЛОГИТ И ПРОБИТ МОДЕЛЕЙ:")
print("-"*60)
print(f"{'Часы':^8} | {'Логит P':^10} | {'Пробит P':^10} | {'Разница':^8} | {'Логит ME':^10} | {'Пробит ME':^10}")
print("-"*60)

for h in test_students:
    p_logit = expit(logit_result.params.iloc[0] + logit_result.params.iloc[1] * h)
    p_probit = stats.norm.cdf(probit_result.params.iloc[0] + probit_result.params.iloc[1] * h)
    me_logit = marginal_effect_logit(h, logit_result.params)[0]
    me_probit = marginal_effect_probit(h, probit_result.params)[0]
    print(f"{h:^8} | {p_logit:^10.3f} | {p_probit:^10.3f} | {p_logit-p_probit:^8.3f} | {me_logit:^10.4f} | {me_probit:^10.4f}")

# Масштабирование коэффициентов
print(f"\nСООТНОШЕНИЕ КОЭФФИЦИЕНТОВ:")
print(f"Логит β₂ / Пробит β₂ = {logit_result.params.iloc[1]/probit_result.params.iloc[1]:.3f}")
print(f"Теоретическое соотношение: √(π²/3) / 1 ≈ 1.814")


#СРАВНЕНИЕ ЛОГИТ И ПРОБИТ МОДЕЛЕЙ

Обе модели предназначены для бинарной классификации и имеют S-образную форму зависимости вероятности от линейного индекса. Их различие заключается в выборе распределения: логит основан на логистическом распределении, пробит — на нормальном.

На практике результаты моделей обычно очень близки. Коэффициенты отличаются масштабом (приблизительно в 1.6 раза), но качественные выводы совпадают. Логит чаще используется в прикладной экономике и машинном обучении из-за более простой интерпретации через шансы, тогда как пробит традиционно применяется в эконометрике и теоретических моделях с латентной нормальной переменной.

In [ ]:
print("\n" + "="*80)
print("ЧАСТЬ 6: ДЕТАЛЬНОЕ СРАВНЕНИЕ ЛОГИТ И ПРОБИТ МОДЕЛЕЙ")
print("="*80)
print("\nВ этой части мы построим 6 графиков, сравнивающих логит и пробит модели.")
print("Каждый график иллюстрирует определенный аспект сходств и различий.")
print("-"*80)

# Подготовка данных для графиков
hours_range = np.linspace(0, 50, 200)
X_range = sm.add_constant(hours_range)

# Предсказания моделей
p_range_logit = logit_result.predict(X_range)
p_range_probit = probit_result.predict(X_range)

# Разница в предсказаниях
pred_diff = p_range_logit - p_range_probit

# Предельные эффекты
me_logit_range = [marginal_effect_logit(h, logit_result.params)[0] for h in hours_range]
me_probit_range = [marginal_effect_probit(h, probit_result.params)[0] for h in hours_range]

# Для QQ-plot
quantiles = np.linspace(0.01, 0.99, 100)
logit_quantiles = -np.log(1/quantiles - 1)  # обратная функция логита
norm_quantiles = stats.norm.ppf(quantiles)

# ============================================================================
# ГРАФИК 1: Сравнение предсказанных вероятностей
# ============================================================================

plt.figure(figsize=(10, 6))
plt.plot(hours_range, p_range_logit, 'b-', linewidth=2.5, label='Логит модель')
plt.plot(hours_range, p_range_probit, 'r--', linewidth=2.5, label='Пробит модель')
plt.scatter(df['hours'], df['passed'], alpha=0.1, s=10, c='gray', label='Исходные данные')
plt.xlabel('Часы подготовки', fontsize=12)
plt.ylabel('Вероятность сдачи P(Y=1)', fontsize=12)
plt.title('ГРАФИК 1: Сравнение предсказанных вероятностей\nлогит vs пробит', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot1_predictions.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 1 (Сравнение предсказанных вероятностей):")
print("-"*80)
print("""
На графике видно, что кривые логит и пробит практически совпадают
на всем диапазоне часов подготовки от 0 до 50.

КЛЮЧЕВОЕ НАБЛЮДЕНИЕ:
• Обе модели дают S-образную форму, что правильно отражает нелинейность
• Визуально различить модели почти невозможно — они накладываются друг на друга
• Это подтверждает тезис из лекции: при значениях аргумента, не слишком
  далеких от нуля, логистическая функция и функция стандартного нормального
  распределения ведут себя сходным образом

ВЫВОД: Для большинства практических задач логит и пробит
дают практически одинаковые предсказания вероятностей.
""")

# ============================================================================
# ГРАФИК 2: Разница в предсказаниях
# ============================================================================

plt.figure(figsize=(10, 6))
plt.plot(hours_range, pred_diff, 'g-', linewidth=2.5, color='purple')
plt.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
plt.axvline(x=15, color='red', linestyle='--', alpha=0.7, label='15 часов (пример из лекции)')
plt.xlabel('Часы подготовки', fontsize=12)
plt.ylabel('Разница (Логит - Пробит)', fontsize=12)
plt.title('ГРАФИК 2: Разница в предсказаниях вероятностей\n(Логит минус Пробит)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot2_difference.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 2 (Разница в предсказаниях):")
print("-"*80)
print(f"""
Этот график показывает разницу между предсказаниями логит и пробит моделей.

КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:
• Разница не равна нулю, но очень мала (в пределах ±0.04)
• Максимальное расхождение наблюдается в области средних значений
• Для студента с 15 часами (пример из лекции) разница составляет
  примерно {pred_diff[abs(hours_range-15).argmin()]:.3f}

ЧИСЛОВЫЕ ЗНАЧЕНИЯ:
• Минимальная разница: {pred_diff.min():.4f}
• Максимальная разница: {pred_diff.max():.4f}
• Средняя абсолютная разница: {np.abs(pred_diff).mean():.4f}

ВЫВОД: Разница между моделями существует, но она настолько мала,
что не влияет на содержательные выводы исследования.
""")

# ============================================================================
# ГРАФИК 3: Сравнение предельных эффектов
# ============================================================================

plt.figure(figsize=(10, 6))
plt.plot(hours_range, me_logit_range, 'b-', linewidth=2.5, label='Логит: предельный эффект')
plt.plot(hours_range, me_probit_range, 'r--', linewidth=2.5, label='Пробит: предельный эффект')
plt.xlabel('Часы подготовки', fontsize=12)
plt.ylabel('Предельный эффект (dp/dx)', fontsize=12)
plt.title('ГРАФИК 3: Сравнение предельных эффектов\nвлияние дополнительного часа подготовки', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot3_marginal_effects.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 3 (Сравнение предельных эффектов):")
print("-"*80)
print("""
График показывает, как меняется влияние дополнительного часа подготовки
в зависимости от того, сколько студент уже занимался.

КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:
• Форма кривых одинакова — обе имеют колоколообразную форму
• Максимальный эффект достигается в одной и той же области (15-20 часов)
• Логит дает чуть более высокий пик предельного эффекта

ИНТЕРПРЕТАЦИЯ ДЛЯ ПРИМЕРА СО СТУДЕНТАМИ:
• 0-10 часов: эффект мал — студент еще не набрал базу
• 15-20 часов: эффект максимален — "золотой час" подготовки
• >30 часов: эффект стремится к нулю — студент уже всё выучил

ВЫВОД: Обе модели правильно отражают нелинейность влияния подготовки,
и их предельные эффекты очень близки по форме и величине.
""")

# ============================================================================
# ГРАФИК 4: Связь z и вероятности для обеих моделей
# ============================================================================

z_range_plot = np.linspace(-8, 8, 200)
p_from_z_logit = expit(z_range_plot)
p_from_z_probit = stats.norm.cdf(z_range_plot)

plt.figure(figsize=(10, 6))
plt.plot(z_range_plot, p_from_z_logit, 'b-', linewidth=2.5, label='Логит: F(z) = 1/(1+e^(-z))')
plt.plot(z_range_plot, p_from_z_probit, 'r--', linewidth=2.5, label='Пробит: Φ(z)')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='z = 0')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z = β₀ + β₁·hours (линейный индекс)', fontsize=12)
plt.ylabel('Вероятность P(Y=1) = F(z)', fontsize=12)
plt.title('ГРАФИК 4: Трансформация линейного индекса z в вероятность\nдля логит и пробит моделей', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot4_z_transform.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 4 (Трансформация z в вероятность):")
print("-"*80)
print("""
Этот график показывает, как разные функции преобразуют линейный индекс z в вероятность.

КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:
• При z = 0 обе функции дают P = 0.5 (точка неопределенности)
• При отрицательных z пробит дает чуть более низкие вероятности
• При положительных z пробит дает чуть более высокие вероятности
• Разница заметна только при |z| > 1

МАТЕМАТИЧЕСКИЙ СМЫСЛ:
• Логит использует логистическую функцию F(z) = 1/(1+e^(-z))
• Пробит использует функцию стандартного нормального распределения Φ(z)
• Обе функции монотонны и отображают R в интервал (0,1)

ВЫВОД: Различия в трансформации существуют, но они невелики
и проявляются в основном на хвостах распределения.
""")

# ============================================================================
# ГРАФИК 5: Весовые функции для предельных эффектов
# ============================================================================

f_z_logit = expit(z_range_plot) * (1 - expit(z_range_plot))
f_z_probit = stats.norm.pdf(z_range_plot)

plt.figure(figsize=(10, 6))
plt.plot(z_range_plot, f_z_logit, 'b-', linewidth=2.5, label='Логит: f(z) = p(1-p)')
plt.plot(z_range_plot, f_z_probit, 'r--', linewidth=2.5, label='Пробит: φ(z)')
plt.fill_between(z_range_plot, f_z_logit, alpha=0.2, color='blue')
plt.fill_between(z_range_plot, f_z_probit, alpha=0.2, color='red')
plt.xlabel('z = β₀ + β₁·hours', fontsize=12)
plt.ylabel('Вес для предельного эффекта f(z)', fontsize=12)
plt.title('ГРАФИК 5: Весовые функции для предельных эффектов\nf(z) = производная функции распределения', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot5_weight_functions.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 5 (Весовые функции для предельных эффектов):")
print("-"*80)
print("""
Этот график показывает плотности распределений — они определяют,
с каким весом коэффициент β₂ влияет на предельный эффект.

КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:
• Логистическая плотность (логит) немного выше в центре и на хвостах
• Нормальная плотность (пробит) более сконцентрирована в центре
• Площадь под обеими кривыми равна 1 (свойство плотности)

СВЯЗЬ С ПРИМЕРОМ:
• При z = -1.5 (15 часов подготовки) вес для логита ≈ {f_z_logit[abs(z_range_plot+1.5).argmin()]:.3f}
• При z = 11 (40 часов подготовки) вес практически равен 0
• Это объясняет, почему предельные эффекты различаются для разных студентов

ВЫВОД: Различия в весовых функциях приводят к тому, что логит
придает больший вес экстремальным значениям (тяжелые хвосты).
""")

# ============================================================================
# ГРАФИК 6: QQ-plot (сравнение квантилей)
# ============================================================================

plt.figure(figsize=(10, 6))
plt.scatter(norm_quantiles, logit_quantiles, alpha=0.7, s=30, c='blue', edgecolors='black')
plt.plot([-4, 4], [-4, 4], 'r--', linewidth=2, label='Линия y=x')
plt.plot([-4, 4], [-4*1.6, 4*1.6], 'g--', linewidth=2, label='Линия y=1.6x (теоретическое соотношение)')
plt.xlabel('Нормальные квантили (Пробит)', fontsize=12)
plt.ylabel('Логистические квантили (Логит)', fontsize=12)
plt.title('ГРАФИК 6: QQ-plot — сравнение квантилей распределений\nлогит и пробит', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('06_comparison_plot6_qqplot.png', dpi=150)
plt.show()

print("\n" + "-"*80)
print("ВЫВОД К ГРАФИКУ 6 (QQ-plot — сравнение квантилей):")
print("-"*80)
print("""
QQ-plot (квантиль-квантиль график) показывает, как связаны квантили
двух распределений — логистического и нормального.

КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:
• Точки ложатся не на линию y=x, а на линию с большим наклоном
• Наклон примерно равен 1.6, что соответствует теоретическому соотношению
• Это визуализирует связь: β_logit ≈ 1.6 · β_probit

ТЕОРЕТИЧЕСКОЕ ОБОСНОВАНИЕ:
• Дисперсия логистического распределения: σ²_logit = π²/3 ≈ 3.29
• Дисперсия стандартного нормального: σ²_probit = 1
• Для сопоставимости масштабов: β_logit / β_probit ≈ √(π²/3) ≈ 1.8
• На практике используют 1.6-1.8 как правило пересчета

ВЫВОД: Коэффициенты логит и пробит несопоставимы напрямую,
но их можно пересчитывать через это соотношение. Предельные эффекты
после масштабирования становятся практически идентичными.
""")

#УСТОЙЧИВОСТЬ К ВЫБРОСАМ

И логит, и пробит-модели менее чувствительны к выбросам по зависимой переменной, поскольку она бинарная. Однако выбросы по объясняющим переменным могут существенно влиять на оценку коэффициентов, так как линейный индекс

𝑥𝑇𝛽

напрямую зависит от значений признаков.

Логит имеет несколько более «тяжёлые хвосты» распределения по сравнению с пробитом, что делает его немного устойчивее к экстремальным значениям линейного индекса. Тем не менее обе модели требуют предварительного анализа данных и при необходимости масштабирования или обработки выбросов.

In [ ]:
print("\n" + "="*80)
print("ЧАСТЬ 7: УСТОЙЧИВОСТЬ К ВЫБРОСАМ")
print("="*80)

# Добавляем выбросы (студенты-экстремалы)
df_outlier = df.copy()
outliers = pd.DataFrame({
    'hours': [100, 0.1, 0.05, 95, 105],
    'passed': [1, 0, 0, 1, 1],
    'p_true': [1.0, 0.0, 0.0, 1.0, 1.0]
})
df_outlier = pd.concat([df_outlier, outliers], ignore_index=True)

print(f"Добавлено 5 экстремальных наблюдений")
print(f"Всего наблюдений: {len(df_outlier)}")
print(f"Максимальные часы: {df_outlier['hours'].max():.1f}")
print(f"Минимальные часы: {df_outlier['hours'].min():.1f}")

# Переоцениваем модели с выбросами
X_out = sm.add_constant(df_outlier['hours'])
y_out = df_outlier['passed']

logit_out = Logit(y_out, X_out).fit(disp=0)
probit_out = Probit(y_out, X_out).fit(disp=0)

print("\nВЛИЯНИЕ ВЫБРОСОВ НА КОЭФФИЦИЕНТЫ:")
print("-"*60)
print(f"{'Модель':<10} | {'Без выбросов β₂':^15} | {'С выбросами β₂':^15} | {'Изменение %':^12}")
print("-"*60)
change_logit = (logit_out.params.iloc[1] - logit_result.params.iloc[1])/logit_result.params.iloc[1]*100
change_probit = (probit_out.params.iloc[1] - probit_result.params.iloc[1])/probit_result.params.iloc[1]*100
print(f"{'Логит':<10} | {logit_result.params.iloc[1]:^15.3f} | {logit_out.params.iloc[1]:^15.3f} | {change_logit:^12.1f}")
print(f"{'Пробит':<10} | {probit_result.params.iloc[1]:^15.3f} | {probit_out.params.iloc[1]:^15.3f} | {change_probit:^12.1f}")

# Визуализация влияния выбросов
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

hours_plot = np.linspace(0, 110, 300)
X_plot = sm.add_constant(hours_plot)

# Логит с выбросами и без
p_logit_clean = logit_result.predict(X_plot)
p_logit_out = logit_out.predict(X_plot)

axes[0].plot(hours_plot, p_logit_clean, 'b-', linewidth=2, label='Логит (без выбросов)')
axes[0].plot(hours_plot, p_logit_out, 'b--', linewidth=2, label='Логит (с выбросами)')
axes[0].scatter(outliers['hours'], outliers['passed'], color='red', s=100,
                marker='X', label='Выбросы', zorder=5)
axes[0].set_xlabel('Часы подготовки')
axes[0].set_ylabel('Вероятность P(Y=1)')
axes[0].set_title('Логит: устойчивость к выбросам')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 110])
# Пробит с выбросами и без
p_probit_clean = probit_result.predict(X_plot)
p_probit_out = probit_out.predict(X_plot)

axes[1].plot(hours_plot, p_probit_clean, 'r-', linewidth=2, label='Пробит (без выбросов)')
axes[1].plot(hours_plot, p_probit_out, 'r--', linewidth=2, label='Пробит (с выбросами)')
axes[1].scatter(outliers['hours'], outliers['passed'], color='red', s=100,
                marker='X', label='Выбросы', zorder=5)
axes[1].set_xlabel('Часы подготовки')
axes[1].set_ylabel('Вероятность P(Y=1)')
axes[1].set_title('Пробит: чувствительность к выбросам')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 110])

plt.suptitle('Сравнение устойчивости логит и пробит моделей к выбросам', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('06_outlier_robustness.png', dpi=150)
plt.show()

print("\nВЫВОД 6 (Устойчивость к выбросам):")
print("- Логит имеет более тяжелые хвосты → лучше справляется с выбросами")
print("- Пробит сильнее реагирует на экстремальные наблюдения")
print("- При наличии выбросов логит является более надежным выбором")


In [ ]:
print("\n" + "="*80)
print("ИТОГОВЫЕ ВЫВОДЫ И РЕКОМЕНДАЦИИ")
print("="*80)

print("""
1. ЛИНЕЙНАЯ МОДЕЛЬ ВЕРОЯТНОСТИ (ЛМВ):
   - Проста в реализации и интерпретации
   - Некорректна для бинарных зависимых переменных
   - Дает вероятности вне [0,1] и постоянные предельные эффекты
   - Использовать только как первое приближение

2. ЛОГИТ МОДЕЛЬ:
   - Основана на логистическом распределении (тяжелые хвосты)
   - Всегда дает вероятности в интервале (0, 1)
   - Нелинейные предельные эффекты (зависят от x)
   - Устойчива к выбросам
   - Легко интерпретируется через отношение шансов

3. ПРОБИТ МОДЕЛЬ:
   - Основана на нормальном распределении (легкие хвосты)
   - Теоретически обоснована (связь с латентной переменной)
   - Предпочтительна для сложных экономических моделей
   - Чувствительна к выбросам
   - Результаты очень близки к логит в центральной части

4. КОГДА ЧТО ВЫБИРАТЬ:
   - Логит: общий случай, наличие выбросов, нужна простая интерпретация
   - Пробит: экономические модели, теоретическая обоснованность, вложенные модели
   - В 95% случаев результаты практически идентичны
""")

# Демонстрация итогового сравнения
fig, ax = plt.subplots(figsize=(10, 6))

models_comparison = {
    'ЛМВ (некорректна)': 0.2,
    'Логит (рекомендуется)': 0.95,
    'Пробит (рекомендуется)': 0.93
}

bars = ax.bar(models_comparison.keys(), models_comparison.values(),
              color=['red', 'blue', 'green'], alpha=0.7)
ax.set_ylim([0, 1])
ax.set_ylabel('Оценка применимости для бинарного выбора')
ax.set_title('Сравнение моделей для бинарного выбора')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{height:.0%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('07_final_comparison.png', dpi=150)
plt.show()

Связь между методом максимального правдоподобия (ММП) и моделями логит и пробит является фундаментальной: ММП — это инструмент (способ оценивания), а логит и пробит — это модели (объекты, которые мы оцениваем).
Исходные данные:
Y
Y (0 или 1) и
X
X.

Выбор модели: Мы решаем, какая функция распределения ошибок лучше подходит (логистическая или нормальная).

Если выбираем логистическую -> строим логит-модель.

Если выбираем нормальную -> строим пробит-модель.

Применение метода: Мы используем ММП, чтобы найти коэффициенты для выбранной модели.

ММП ищет коэффициенты
β, которые максимизируют логарифм функции правдоподобия.

Вид функции правдоподобия зависит от того, какую модель мы выбрали на шаге 2 (логит или пробит).

Результат: Мы получаем оцененные коэффициенты
β, которые гарантированно (при больших выборках) являются наилучшими (состоятельными и эффективными) именно для нашей модели.

Логит и пробит — это разные способы задать форму кривой (разные
F
F), а ММП — это универсальный компас, который позволяет найти наилучшее положение этой кривой относительно имеющихся данных.



In [ ]:

plt.rcParams['font.family'] = 'DejaVu Sans'
# ===============
# 1. Генерация данных (пример со студентами)
# ===============
np.random.seed(42)
n_samples = 50

# Часы подготовки (от 10 до 30 часов)
X = np.random.uniform(10, 30, n_samples)
X = X - X.mean()  # Центрируем для лучшей сходимости

# Истинные коэффициенты (которые мы будем искать)
true_beta_1 = -1.5
true_beta_2 = 0.2

# Расчет истинной вероятности через логит
z_true = true_beta_1 + true_beta_2 * X
p_true = 1 / (1 + np.exp(-z_true))

# Генерация бинарного Y (сдал/не сдал) на основе истинных вероятностей
Y = np.random.binomial(1, p_true)

# ===============
# 2. Функции правдоподобия для логита и пробита
# ===============
def log_likelihood_logit(beta, X, Y):
    """Логарифм функции правдоподобия для логит-модели"""
    beta_1, beta_2 = beta
    z = beta_1 + beta_2 * X
    p = 1 / (1 + np.exp(-z))
    # Защита от log(0)
    p = np.clip(p, 1e-10, 1 - 1e-10)
    # Формула: sum( Y*log(p) + (1-Y)*log(1-p) )
    ll = np.sum(Y * np.log(p) + (1 - Y) * np.log(1 - p))
    return -ll  # Возвращаем отрицательное, так как оптимизаторы минимизируют

def log_likelihood_probit(beta, X, Y):
    """Логарифм функции правдоподобия для пробит-модели"""
    beta_1, beta_2 = beta
    z = beta_1 + beta_2 * X
    p = norm.cdf(z)  # Функция стандартного нормального распределения
    p = np.clip(p, 1e-10, 1 - 1e-10)
    ll = np.sum(Y * np.log(p) + (1 - Y) * np.log(1 - p))
    return -ll

# ===============
# 3. Создание сетки коэффициентов для визуализации
# ===============
beta_1_range = np.linspace(-3, 0.5, 100)
beta_2_range = np.linspace(0, 0.4, 100)

# Создаем матрицы для хранения значений функции правдоподобия
ll_logit_matrix = np.zeros((len(beta_1_range), len(beta_2_range)))
ll_probit_matrix = np.zeros((len(beta_1_range), len(beta_2_range)))

# Заполняем матрицы
for i, b1 in enumerate(beta_1_range):
    for j, b2 in enumerate(beta_2_range):
        ll_logit_matrix[i, j] = -log_likelihood_logit([b1, b2], X, Y)
        ll_probit_matrix[i, j] = -log_likelihood_probit([b1, b2], X, Y)

# Находим оптимальные коэффициенты через ММП (для проверки)
result_logit = minimize(log_likelihood_logit, [0, 0], args=(X, Y), method='BFGS')
result_probit = minimize(log_likelihood_probit, [0, 0], args=(X, Y), method='BFGS')

opt_b1_logit, opt_b2_logit = result_logit.x
opt_b1_probit, opt_b2_probit = result_probit.x

# ===============
# 4. ВИЗУАЛИЗАЦИЯ 1: Связь X, Y и подобранной кривой
# ===============
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Сортируем X для гладкой линии
X_sorted = np.sort(X)

# Логит
p_logit = 1 / (1 + np.exp(-(opt_b1_logit + opt_b2_logit * X_sorted)))
axes[0].scatter(X, Y, alpha=0.6, label='Данные (0 - не сдал, 1 - сдал)')
axes[0].plot(X_sorted, p_logit, 'r-', linewidth=2, label=f'Логит (ММП): p = 1/(1+e^-({opt_b1_logit:.2f}+{opt_b2_logit:.2f}x))')
axes[0].set_xlabel('Часы подготовки (X, центрированные)')
axes[0].set_ylabel('Вероятность сдачи P(Y=1)')
axes[0].set_title('Логит-модель: подгонка кривой к данным')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylim(-0.05, 1.05)

# Пробит
p_probit = norm.cdf(opt_b1_probit + opt_b2_probit * X_sorted)
axes[1].scatter(X, Y, alpha=0.6, label='Данные (0 - не сдал, 1 - сдал)')
axes[1].plot(X_sorted, p_probit, 'b-', linewidth=2, label=f'Пробит (ММП): p = Φ({opt_b1_probit:.2f}+{opt_b2_probit:.2f}x)')
axes[1].set_xlabel('Часы подготовки (X, центрированные)')
axes[1].set_ylabel('Вероятность сдачи P(Y=1)')
axes[1].set_title('Пробит-модель: подгонка кривой к данным')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylim(-0.05, 1.05)

plt.suptitle('Рисунок 1: Как модели бинарного выбора подбираются под данные', fontsize=14)
plt.tight_layout()
plt.show()

Интерпретация Рисунка 1
На этом графике показан результат работы ММП. Синие и красные точки — это наши студенты (Y=1 сдали, Y=0 не сдали). Кривые — это подобранные модели.

ММП подобрал коэффициенты так, что кривая логистической (или нормальной) функции проходит максимально близко к 1 там, где Y=1, и близко к 0 там, где Y=0.

Это не прямая линия (как в МНК), а S-образная кривая, которая никогда не выходит за пределы [0, 1].

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Определяем единый размер сетки
n_points = 50  # Количество точек для каждого коэффициента

# СОЗДАЕМ СЕТКУ ЗНАЧЕНИЙ
beta_1_range = np.linspace(-15, 0, n_points)    # 50 точек от -15 до 0 для β₁
beta_2_range = np.linspace(0, 1, n_points)      # 50 точек от 0 до 1 для β₂
B1, B2 = np.meshgrid(beta_1_range, beta_2_range)

# УБЕДИМСЯ, ЧТО МАТРИЦЫ ПРАВДОПОДОБИЯ ИМЕЮТ ПРАВИЛЬНЫЙ РАЗМЕР
# Если они уже созданы, проверьте их размер:
print(f"Размер B1: {B1.shape}")
print(f"Размер ll_logit_matrix: {ll_logit_matrix.shape}")
print(f"Размер ll_logit_matrix.T: {ll_logit_matrix.T.shape}")

# Если ll_logit_matrix имеет размер (100, 100), пересоздайте её с правильным размером:
if ll_logit_matrix.shape != (n_points, n_points):
    print("Пересоздаем матрицы правдоподобия...")
    ll_logit_matrix = np.zeros((n_points, n_points))
    ll_probit_matrix = np.zeros((n_points, n_points))

    for i, b1 in enumerate(beta_1_range):
        for j, b2 in enumerate(beta_2_range):
            ll_logit_matrix[i, j] = -log_likelihood_logit([b1, b2], X, Y)
            ll_probit_matrix[i, j] = -log_likelihood_probit([b1, b2], X, Y)

# Создаем фигуру
fig = plt.figure(figsize=(14, 6))

# Логит с контурами
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(B1, B2, ll_logit_matrix.T, cmap='viridis', alpha=0.7)

# Добавим контурные линии на плоскость Z=-40
ax1.contour(B1, B2, ll_logit_matrix.T, zdir='z', offset=-40, cmap='viridis', alpha=0.5)

# Отметим максимум
ax1.scatter(opt_b1_logit, opt_b2_logit, -result_logit.fun,
    color='red', s=100, label=f'Максимум ММП\nβ₁={opt_b1_logit:.2f}\nβ₂={opt_b2_logit:.2f}')

ax1.set_xlabel('β₁ (свободный член)')
ax1.set_ylabel('β₂ (влияние подготовки)')
ax1.set_zlabel('Log-правдоподобие')
ax1.set_title('Логит: 3D поверхность с контурами')
ax1.legend()

# Пробит
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(B1, B2, ll_probit_matrix.T, cmap='plasma', alpha=0.7)
ax2.contour(B1, B2, ll_probit_matrix.T, zdir='z', offset=-40, cmap='plasma', alpha=0.5)
ax2.scatter(opt_b1_probit, opt_b2_probit, -result_probit.fun,
    color='red', s=100, label=f'Максимум ММП\nβ₁={opt_b1_probit:.2f}\nβ₂={opt_b2_probit:.2f}')
ax2.set_xlabel('β₁ (свободный член)')
ax2.set_ylabel('β₂ (влияние подготовки)')
ax2.set_zlabel('Log-правдоподобие')
ax2.set_title('Пробит: 3D поверхность с контурами')
ax2.legend()

plt.tight_layout()
plt.show()

ММП успешно нашел глобальный максимум - красная точка четко выделяется на вершине холма

Модель идентифицируема - уникальное решение существует и найдено

Влияние факторов значимо - β₂ значительно отличается от нуля (пик не на краю по оси β₂)

Логит и пробит дают схожие результаты - форма поверхностей почти идентична

Точность оценок - по крутизне склонов видно, что β₂ оценивается точнее, чем β₁

Вопросы для самопроверки

Вопрос 1:
Почему нельзя использовать обычную линейную регрессию для задачи бинарной классификации? В чём её принципиальная проблема?

Ответ: линейная регрессия выдаёт прогноз на всей числовой прямой, поэтому для бинарного исхода может получить значения меньше 0 или больше 1, что невозможно интерпретировать как вероятность. Кроме того, она предполагает постоянный предельный эффект признака и не учитывает S-образную зависимость вероятности от факторов.

Вопрос 2:
Зачем в логит- и пробит-моделях используется S-образная функция? Какие свойства этой функции важны?

Ответ: S-образная функция переводит линейный индекс модели в вероятность из интервала от 0 до 1. Важны монотонность, ограниченность и нелинейность: при средних значениях вероятность меняется быстрее, а около 0 и 1 эффект признаков затухает.

Вопрос 3:
Что такое log-odds в логит-модели? Как интерпретировать коэффициенты при признаках?

Ответ: log-odds — это логарифм отношения шансов события к шансам его отсутствия: log(p / (1 - p)). Коэффициент при признаке показывает, на сколько изменится log-odds при увеличении признака на единицу. Если коэффициент возвести в экспоненту, получится мультипликативное изменение odds.

Вопрос 4:
В чём концептуальное различие между логитом и пробитом? Почему их результаты на практике часто оказываются похожими?

Ответ: логит использует логистическую функцию распределения, а пробит — функцию стандартного нормального распределения. Обе функции имеют похожую S-образную форму, особенно в центральной области, поэтому предсказанные вероятности и выводы часто почти совпадают; обычно отличается только масштаб коэффициентов.

Вопрос 5:
Что такое латентная переменная в пробит-модели? Как она связана с наблюдаемым бинарным исходом?

Ответ: латентная переменная — это скрытая непрерывная склонность к событию, которую мы напрямую не наблюдаем. В пробит-модели предполагается, что бинарный исход равен 1, если эта скрытая переменная превышает порог, и 0 иначе; нормальное распределение ошибки приводит к probit-связи.

Вопрос 6:
Почему параметры логит- и пробит-моделей нельзя оценить методом наименьших квадратов? Какой метод используется вместо этого и почему?

Ответ: для бинарного исхода ошибки не являются нормально распределёнными с постоянной дисперсией, а зависимость вероятности от признаков нелинейна. Поэтому МНК плохо соответствует вероятностной природе задачи. Вместо него используют метод максимального правдоподобия: выбирают параметры, при которых наблюдаемые 0 и 1 имеют максимальную вероятность.

Вопрос 7:
Как меняется вероятность события при увеличении признака на единицу в логит-модели? Постоянен ли этот эффект?

Ответ: вероятность меняется на предельный эффект, который зависит от текущего значения признаков и самой вероятности: для логита он пропорционален beta_j * p(x) * (1 - p(x)). Эффект не постоянен: он максимален около p = 0.5 и меньше около вероятностей, близких к 0 или 1.

Вопрос 8:
В каких случаях логит удобнее пробита, а в каких — наоборот?

Ответ: логит удобнее, когда нужна простая прикладная интерпретация через odds и odds ratio, а также в большинстве стандартных задач бинарной классификации. Пробит чаще выбирают в эконометрике и моделях с латентной нормальной переменной, где нормальность ошибки важна теоретически. На практике стоит сравнить качество моделей, потому что их предсказания часто очень близки.
